**فراخوانی فایل و نصب کتابخانه های مورد نیاز**

In [10]:
from os import listdir
from os.path import isfile, join,isdir
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize
import json
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
import nltk
nltk.download('stopwords')
nltk.download('punkt')
cachedStopWords = stopwords.words("english")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


**پیش پردازش متن و ایجاد فایل برای ترین**

In [11]:
path = '/content/drive/MyDrive/deception'
#path=''
columns = ['sent', 'class']  
stemmer = SnowballStemmer("english")
def removeStopWords(text):
    text = ' '.join([word for word in text.split() if word not in cachedStopWords])
    return text
def stemmWords(text):
    text = " ".join([stemmer.stem(word) for word in text.split(" ")])
    return text
#Function that trains a Naive Bayes classifier
def trainNaiveBayes(fileList, stopWords=False, stemm = False):
    rows = []
    for toCheckFile in fileList:
        file = open(path + '/' + toCheckFile,"r", encoding="UTF8")
        file = file.read()
        cl = 'true'
        if ('lie' in toCheckFile):
            cl = 'lie'
        if(stopWords):
            file = removeStopWords(file)
            
        if(stemm):
            file = stemmWords(file)            
        rows.append([file, cl])  
        
    trainingData = pd.DataFrame(rows, columns=columns)  
    # get tdm of  lie
    lieDocs = [row['sent'] for index,row in trainingData.iterrows() if row['class'] == 'lie']
    vec_lie = CountVectorizer()
    X_lie = vec_lie.fit_transform(lieDocs)

    #get tdm of true
    trueDocs = [row['sent'] for index,row in trainingData.iterrows() if row['class'] == 'true']
    vec_true = CountVectorizer()
    X_true = vec_true.fit_transform(trueDocs)
    
    #get freq of each word in lie and true class
    word_list_true = vec_true.get_feature_names_out();    
    count_list_true = X_true.toarray().sum(axis=0) 
    freq_true = dict(zip(word_list_true,count_list_true))
    word_list_lie = vec_lie.get_feature_names_out();    
    count_list_lie = X_lie.toarray().sum(axis=0) 
    freq_lie = dict(zip(word_list_lie,count_list_lie))

    #get prob of each word in class
    prob_lie = []
    for word, count in zip(word_list_lie, count_list_lie):
        prob_lie.append(count/len(word_list_lie))
    probsLie = dict(zip(word_list_lie, prob_lie))

    prob_true = []
    for word, count in zip(word_list_true, count_list_true):
        prob_true.append(count/len(word_list_true))
    probsTrue = dict(zip(word_list_true, prob_true))


    docs = [row['sent'] for index,row in trainingData.iterrows()]

    vec = CountVectorizer()
    X = vec.fit_transform(docs)

    total_features = len(vec.get_feature_names_out())
    total_cnts_features_true = count_list_true.sum(axis=0)
    total_cnts_features_lie = count_list_lie.sum(axis=0)
    return {
        "propTrue":freq_true,
        "propLie": freq_lie,
        "totalFeatures": total_features,
        "totalFeaturesTrue":total_cnts_features_true,
        "totalFeaturesLie":total_cnts_features_lie
    }

**تابع** **تست**

In [12]:
def testNaiveBayes(file,trainData,stopWords = False, stemm = False):
        #prop new sent in statement class.
    testFile = open(path + '/' + file,"r", encoding="UTF8") 
    testFile = testFile.read()
    if(stopWords):
        testFile = removeStopWords(testFile)
        
    if(stemm):
        testFile = stemmWords(testFile)             
    testFileList = word_tokenize(testFile)
    #prob new file in true class
    probTrueWithTestFile = []
    for word in testFileList:
        if word in trainData["propTrue"].keys():
            count = trainData["propTrue"][word]
        else:
            count = 0
        probTrueWithTestFile.append((count + 1)/(trainData["totalFeaturesTrue"] + trainData["totalFeatures"]))
    propTrue = dict(zip(testFileList,probTrueWithTestFile))
                                    
    #prob new file in lie class
    probLieWithTestFile = []
    for word in testFileList:
        if word in trainData["propLie"].keys():
            count = trainData["propLie"][word]
        else:
            count = 0
        probLieWithTestFile.append((count + 1)/(trainData["totalFeaturesLie"] + trainData["totalFeatures"]))
    propLie = dict(zip(testFileList,probLieWithTestFile))
    
    #cacl mult prob

    predLie = 1
    predTrue = 1
    for k, v in propLie.items():
        predLie = predLie*v 
    for k, v in propTrue.items():
        predTrue = predTrue*v 
    predLie = predLie * 0.5
    predTrue = predTrue * 0.5
    resutl = []
    if predTrue> predLie:
        result =  ['true',file]
    else:
        result = ['lie',file]
    
    isCorrect = True
    if(result[0] not in file):
        isCorrect = False
    return {"result" : result, "isCorrect" : isCorrect}

**خواندن فایل ها و ارسال برای توابع یادگیری و تست و در نهایت ایجاد خروجی**

In [13]:

def getFiles():
    #get folder name as user input
    if(isdir(path) == False):
        getFiles()
    files = [f for f in listdir(path) if isfile(join(path, f))]    
    return files

#get given folder files list    
path = input("Please enter the name of folder:\n")
files = getFiles()
print("loading...","\n")
#test 196 files and train 195 each time
checkNormal =[]
correctNormal = 0
correctStopWords = 0
correctStemmed = 0
correctStemmStopWords = 0
for file in files:
    temp = files
    temp.remove(file)
    rows = trainNaiveBayes(temp)
    test = testNaiveBayes(file,rows)
    trainStopWords = trainNaiveBayes(temp,stopWords = True)
    testStopWords = testNaiveBayes(file,trainStopWords,stopWords = True)
    trainStemmed = trainNaiveBayes(temp,stemm = True)
    testStemmed = testNaiveBayes(file,trainStemmed,stemm = True)    
    trainStemmStopWords = trainNaiveBayes(temp,stemm = True,stopWords = True)
    testStemmStopWords = testNaiveBayes(file,trainStemmStopWords,stemm = True,stopWords = True)    
    if(test['isCorrect']):
        correctNormal += 1
    if(testStopWords['isCorrect']):
        correctStopWords += 1
    if(testStemmed['isCorrect']):
        correctStemmed += 1        
    if(testStemmStopWords['isCorrect']):
        correctStemmStopWords += 1    
#check Accuracy 
accNormal = (correctNormal / len(files)) * 100
accStopWords = (correctStopWords / len(files)) * 100
accStemmed = (correctStemmed / len(files)) * 100
accStemmStopWords = (correctStemmStopWords / len(files)) * 100

file = open("answers.txt","w", encoding="UTF8")
result = [
    "Normal Accuracy:" + str(accNormal),
    "Removed Stopwords Accuracy:" + str(accStopWords),
    "Stemmed words Accuracy:"  + str(accStemmed),
    "Stemmed words and removed stopwords Accuracy:" + str(accStemmStopWords),
]
file.write(",".join(result))
file.close()
print("Done!. result stored in answers.txt")

Please enter the name of folder:
/content/drive/MyDrive/deception
loading... 

Done!. result stored in answers.txt


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
